# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsserGharib1/flyrank-internshipML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup and access

Use the gated warehouse through a Colab secret named `HF_TOKEN`. March 2026 is the feature month and April 2026 is the outcome month. June stays sealed because the `_sample` table is the final month, not a random sample.

In [2]:
%pip -q install duckdb pandas scikit-learn

import os, duckdb, pandas as pd, numpy as np

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "Add a Hugging Face read token as a Colab secret named HF_TOKEN."

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = BASE + "/fact_content_daily_performance"
FEATURE_MONTH = "2026-03"
OUTCOME_MONTH = "2026-04"

def month_rel(month):
    return f"read_parquet('{FACT}/month={month}/*.parquet')"

print("Feature month:", FEATURE_MONTH, "| Outcome month:", OUTCOME_MONTH)

Feature month: 2026-03 | Outcome month: 2026-04


## 1. Unit of analysis + time window

**One row = one content item for one client at the 1 April 2026 decision point.** March supplies features. April supplies the later outcome. The daily warehouse table is aggregated to this content-level grain before modeling.

## 2. Fields: feature / label / context / excluded

**Contract in five answers**

1. One row is one content item for one client at the decision point.
2. The main source is `fact_content_daily_performance`. `dim_clients` is used only for the history-coverage limitation.
3. Features use March 2026. The outcome uses April 2026.
4. The target is whether April-vs-March impression change is at least 20 percentage points below the median change for that client.
5. GA4 fields and `fact_content_query_90d` are excluded. Query 3 measures GA4 availability with `IS TRUE`. The fixed query table window is not aligned to this decision point.

**Features:** March impressions, clicks, active days, average position, and within-March momentum. **Context:** pseudonymized client/content IDs and dates. **Label-only fields:** April impressions and anything derived from them.

## 3. Verify it with queries

The three required verification checks cover grain, size/date span, and availability. After those checks I build the five-feature frame and run one deliberate leakage test.

In [3]:
# QUERY 1 of 3 — verify daily client/content grain.
grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_in_group
    FROM {month_rel(FEATURE_MONTH)}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("QUERY 1 — duplicate grain groups:", len(grain))

QUERY 1 — duplicate grain groups: 0


In [4]:
# QUERY 2 of 3 — row count and date span for the feature month.
span = con.sql(f"""
    SELECT COUNT(*) AS rows_in_month,
           COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS content_items,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM {month_rel(FEATURE_MONTH)}
""").df()

print("QUERY 2 — March slice")
print(span.to_string(index=False))

QUERY 2 — March slice
 rows_in_month  clients  content_items first_date  last_date
       9841378       55         331437 2026-03-01 2026-03-31


In [5]:
# QUERY 3 of 3 — availability, using the release flag exactly as documented.
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_analytics,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS rows_with_impressions
    FROM {month_rel(FEATURE_MONTH)}
""").df()

print("QUERY 3 — availability")
print(avail.to_string(index=False))
a = avail.iloc[0]
print(f"GA4 available (IS TRUE): {a.rows_with_analytics / a.total_rows:.1%}")
print(f"Rows with GSC impressions: {a.rows_with_impressions / a.total_rows:.1%}")

QUERY 3 — availability
 total_rows  rows_with_analytics  rows_with_impressions
    9841378               413966                3611061
GA4 available (IS TRUE): 4.2%
Rows with GSC impressions: 36.7%


### The five features

- `impressions_prev30` — available on 1 April because it sums March impressions only.
- `clicks_prev30` — available on 1 April because it sums March clicks only.
- `active_days_prev30` — available on 1 April because it counts March days with impressions.
- `avg_position_prev30` — available on 1 April because its impression-weighted calculation uses only March `gsc_avg_position` and March impressions.
- `momentum_in_march` — available on 1 April because it compares average daily impressions from March 1–15 with March 16–31.

GA4 is excluded because the saved check shows only a small share of March rows with `ga4_data_available IS TRUE`. I do not interpret the remaining rows as measured zero engagement.

In [6]:
# Build the frame. March gives me the features, April gives me the outcome.

FLOOR = 100   # provisional minimum-support floor carried from the framing step

frame = con.sql(f"""
    WITH march AS (
        SELECT client_hash_id  AS client_id,
               content_hash_id AS content_id,
               SUM(gsc_impressions)                    AS impressions_prev30,
               SUM(gsc_clicks)                         AS clicks_prev30,
               COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days_prev30,
               -- impression-weighted monthly position from the warehouse position column.
               SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE gsc_avg_position > 0) * 1.0
                 / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_avg_position > 0), 0)
                 AS avg_position_prev30,
               SUM(gsc_impressions) FILTER (WHERE EXTRACT(day FROM report_date) > 15)  AS late_march,
               SUM(gsc_impressions) FILTER (WHERE EXTRACT(day FROM report_date) <= 15) AS early_march
        FROM {month_rel(FEATURE_MONTH)}
        GROUP BY 1, 2
    ),
    april AS (
        SELECT client_hash_id  AS client_id,
               content_hash_id AS content_id,
               SUM(gsc_impressions) AS impressions_next30
        FROM {month_rel(OUTCOME_MONTH)}
        GROUP BY 1, 2
    )
    SELECT m.client_id,
           m.content_id,
           m.impressions_prev30,
           m.clicks_prev30,
           m.active_days_prev30,
           m.avg_position_prev30,
           CASE WHEN m.early_march > 0
                THEN ((m.late_march / 16.0) - (m.early_march / 15.0)) * 100.0
                     / (m.early_march / 15.0)
                ELSE NULL END              AS momentum_in_march,
           COALESCE(a.impressions_next30, 0) AS impressions_next30
    FROM march m
    LEFT JOIN april a USING (client_id, content_id)
    WHERE m.impressions_prev30 >= {FLOOR}
    -- sorted so a rerun reads the rows in the same order and gives the same score
    ORDER BY client_id, content_id
""").df()

print(f"{len(frame):,} pages from {frame.client_id.nunique()} clients cleared the {FLOOR} impression floor")
unit_rows = frame[["client_id", "content_id"]].drop_duplicates().shape[0]
print(f"rows per client/content item: {len(frame)/unit_rows:.2f}  (1.00 means the grain survived the join)")
assert len(frame) == unit_rows, "The feature frame must contain one row per client/content item."
print()
print(frame.head(5).to_string(index=False))
print()

# Missing values, which the section title asks about.
print("Which columns go blank, and is it random?")
for col in ["impressions_prev30", "clicks_prev30", "active_days_prev30",
            "avg_position_prev30", "momentum_in_march"]:
    print(f"  {col:<22} blank on {frame[col].isna().mean():6.2%} of rows")

blank = frame[frame.momentum_in_march.isna()]
print()
print(f"Only momentum goes blank, on {len(blank):,} pages.")
print("Checking whether those pages look different from the rest:")
print(f"  median March impressions, blank pages : {blank.impressions_prev30.median():,.0f}")
print(f"  median March impressions, the rest    : {frame[frame.momentum_in_march.notna()].impressions_prev30.median():,.0f}")
print(f"  median active days, blank pages       : {blank.active_days_prev30.median():.0f}")
print(f"  median active days, the rest          : {frame[frame.momentum_in_march.notna()].active_days_prev30.median():.0f}")

101,441 pages from 44 clients cleared the 100 impression floor
rows per client/content item: 1.00  (1.00 means the grain survived the join)

              client_id               content_id  impressions_prev30  clicks_prev30  active_days_prev30  avg_position_prev30  momentum_in_march  impressions_next30
client_0797ff3a1fc9a6a5 content_04c67f3541177192               331.0            2.0                  31            14.377644          67.016807               561.0
client_0797ff3a1fc9a6a5 content_0f30e04e709c7b5d               145.0            0.0                  30             8.124138          -4.947917                23.0
client_0797ff3a1fc9a6a5 content_1207efddce873942               461.0            0.0                  31            14.488069         -41.033569               964.0
client_0797ff3a1fc9a6a5 content_167472cd0802a8f3               232.0            0.0                  29            11.961207         135.795455               248.0
client_0797ff3a1fc9a6a5 content_27f8100

In [7]:
# Build the later outcome and the client-relative target.
frame["change_pct"] = (
    (frame["impressions_next30"] - frame["impressions_prev30"])
    / frame["impressions_prev30"] * 100
)
frame["gap_vs_client"] = (
    frame["change_pct"] - frame.groupby("client_id")["change_pct"].transform("median")
)
GAP_CUTOFF = -20
frame["target_falls_behind"] = (frame["gap_vs_client"] <= GAP_CUTOFF).astype(int)

client_medians = frame.groupby("client_id")["change_pct"].median()
print(f"Median page change: {frame.change_pct.median():.1f}%")
print(f"Target base rate at {GAP_CUTOFF} points: {frame.target_falls_behind.mean():.3f}")
print(f"Client-median range: {client_medians.min():.1f}% to {client_medians.max():.1f}%")

Median page change: -22.1%
Target base rate at -20 points: 0.290
Client-median range: -100.0% to 696.0%


`momentum_in_march` is the only one of the five features that can be missing. It is undefined when the first half of March has no impressions, so dropping those rows changes the population. The leakage check below reports the population it actually scores.

### Deliberate leakage test

Add April impressions once as an intentionally invalid feature, score the same held-out clients with and without it, then remove it. April impressions help define the label, so they can never be a model input.

In [8]:
# Score twice: once with the leaked column present, once without. Whole clients held out both times.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

HONEST_FEATURES = ["impressions_prev30", "clicks_prev30", "active_days_prev30",
                   "avg_position_prev30", "momentum_in_march"]
LEAKY_FEATURE = "impressions_next30"   # the label is calculated from this column

model_df = frame.dropna(subset=HONEST_FEATURES).copy()
y = model_df.target_falls_behind
groups = model_df.client_id

train_idx, test_idx = next(
    GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0).split(model_df, y, groups)
)

def quick_score(columns):
    X = model_df[columns]
    m = RandomForestClassifier(n_estimators=120, min_samples_leaf=20, random_state=0, n_jobs=1)
    m.fit(X.iloc[train_idx], y.iloc[train_idx])
    return roc_auc_score(y.iloc[test_idx], m.predict_proba(X.iloc[test_idx])[:, 1])

print(f"{len(model_df):,} rows, {groups.iloc[test_idx].nunique()} clients held out for testing")
print(f"base rate {y.mean():.3f}")
print()

leaked = quick_score(HONEST_FEATURES + [LEAKY_FEATURE])
print(f"with April impressions in there : {leaked:.3f}")

honest = quick_score(HONEST_FEATURES)
print(f"without it                      : {honest:.3f}")
print()
print(f"Leakage inflated AUC by {leaked - honest:+.3f}; only the honest score is relevant.")

model_df = model_df.drop(columns=[LEAKY_FEATURE])
print(f"Dropped it. Still in the frame? {LEAKY_FEATURE in model_df.columns}")
print()
print("Keeping these five for the modelling weeks:")
for f in HONEST_FEATURES:
    print("   ", f)

96,671 rows, 12 clients held out for testing
base rate 0.298

with April impressions in there : 0.923
without it                      : 0.667

Leakage inflated AUC by +0.256; only the honest score is relevant.
Dropped it. Still in the frame? False

Keeping these five for the modelling weeks:
    impressions_prev30
    clicks_prev30
    active_days_prev30
    avg_position_prev30
    momentum_in_march


The saved run shows AUC around 0.92 with the leaked April field and around 0.67 after removing it, a gap of about 0.26. The lower score is the only one carried forward. This is a leakage diagnostic, not the Week-5 model result.

## 4. Data limits

Client history is unbalanced. Different clients began Search Console tracking at different dates, so one shared calendar month does not give every client the same history depth. The check below is scoped to clients that actually appear in this modeling frame.

The data also cannot identify why a page changed or prove that editing caused a later result. Those claims stay outside the project.

In [9]:
# Scope the history check to clients in this modeling frame.
frame_clients = frame[["client_id"]].drop_duplicates()
con.register("frame_clients", frame_clients)

hist = con.sql(f"""
    SELECT COUNT(*) AS clients_with_start,
           COUNT(*) FILTER (WHERE c.gsc_data_start > DATE '2025-09-01') AS started_after_sep_2025,
           MIN(c.gsc_data_start) AS earliest_start,
           MAX(c.gsc_data_start) AS latest_start
    FROM read_parquet('{BASE}/dim_clients.parquet') c
    JOIN frame_clients f ON c.client_hash_id = f.client_id
    WHERE c.gsc_data_start IS NOT NULL
""").df()

print("History coverage for clients in the modeling frame")
print(hist.to_string(index=False))

History coverage for clients in the modeling frame
 clients_with_start  started_after_sep_2025 earliest_start latest_start
                 44                      29     2025-01-27   2026-03-27


This result now describes the same client population used by the frame. I use it only to document unequal history depth, not to claim that short history caused any page-level outcome.

## Self-check

- [x] Five contract answers are explicit
- [x] Exactly three verification checks cover grain, size/date span, and `IS TRUE` availability
- [x] The feature frame has five decision-time features
- [x] One deliberate label-derived leak is shown, scored, and removed
- [x] The named limitation is scoped to the actual modeling clients
- [ ] Final corrected notebook committed and repo URL submitted on the ML-04 card